# 03. HUẤN LUYỆN VÀ ĐÁNH GIÁ 5 MÔ HÌNH MACHINE LEARNING - NUTRIDSS
> **Mục tiêu:** Xây dựng, huấn luyện và so sánh **5 thuật toán Machine Learning** bắt buộc theo yêu cầu tiểu luận:
> 1. **Linear Regression**
> 2. **K-Nearest Neighbors (KNN Regressor)**
> 3. **Decision Tree Regressor**
> 4. **Random Forest Regressor**
> 5. **Artificial Neural Network (ANN - MLPRegressor)**
>
> **Target:** Dự đoán điểm số phù hợp của bữa ăn (`compatibility_rating` từ 1.0 đến 5.0).
> **Metrics đánh giá:** $R^2$ Score, MAE (Mean Absolute Error), RMSE (Root Mean Squared Error).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor

## 1. Nạp và chuẩn bị dữ liệu huấn luyện

In [ ]:
df_ml = pd.read_csv('../data/processed/survey_scenarios_dataset.csv')
print("Kich thuoc dataset ML:", df_ml.shape)
df_ml.head()

## 2. Phân chia Features (X) và Target (y) & Split Train/Test

In [ ]:
feature_cols = ['goal', 'budget_vnd', 'calories', 'protein_g', 'carb_g', 'fat_g', 'estimated_cost_vnd', 'cost_ratio']
X = df_ml[feature_cols]
y = df_ml['compatibility_rating']

cat_cols = ['goal']
num_cols = ['budget_vnd', 'calories', 'protein_g', 'carb_g', 'fat_g', 'estimated_cost_vnd', 'cost_ratio']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first'), cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Kich thuoc X_train: {X_train.shape}, X_test: {X_test.shape}")

## 3. Huấn luyện 5 Mô hình & Đánh giá Chỉ số (Evaluation Metrics)

In [ ]:
models = {
    "Linear Regression": LinearRegression(),
    "KNN Regressor": KNeighborsRegressor(n_neighbors=5),
    "Decision Tree": DecisionTreeRegressor(max_depth=6, random_state=42),
    "Random Forest": RandomForestRegressor(n_estimators=100, max_depth=8, random_state=42),
    "ANN (MLPRegressor)": MLPRegressor(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)
}

results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    r2 = r2_score(y_test, y_pred)
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    
    results.append({
        "Model": name,
        "R2 Score": round(r2, 4),
        "MAE": round(mae, 4),
        "RMSE": round(rmse, 4),
        "Pipeline": pipe
    })

df_results = pd.DataFrame(results)[["Model", "R2 Score", "MAE", "RMSE"]]
df_results.sort_values(by="R2 Score", ascending=False, inplace=True)
df_results

## 4. Trực quan hóa So sánh Hiệu năng 5 Mô hình ML

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

sns.barplot(data=df_results, x='R2 Score', y='Model', ax=axes[0], palette='Blues_r')
axes[0].set_title('So sanh R2 Score (Cang cao cang tot)')

sns.barplot(data=df_results, x='MAE', y='Model', ax=axes[1], palette='Reds_r')
axes[1].set_title('So sanh MAE (Cang thap cang tot)')

sns.barplot(data=df_results, x='RMSE', y='Model', ax=axes[2], palette='Oranges_r')
axes[2].set_title('So sanh RMSE (Cang thap cang tot)')

plt.tight_layout()
os.makedirs('../docs', exist_ok=True)
plt.savefig('../docs/ml_models_performance_comparison.png', dpi=300)
plt.show()

## 5. Xuất Mô hình Tốt nhất (Best Model Export) vào thư mục `models/`

In [ ]:
best_model_info = max(results, key=lambda x: x["R2 Score"])
best_pipeline = best_model_info["Pipeline"]
print(f"Mo hinh tot nhat: {best_model_info['Model']} voi R2 Score = {best_model_info['R2 Score']}")

os.makedirs("../models", exist_ok=True)
joblib.dump(best_pipeline, "../models/best_recipe_ranker.joblib")
print("Da xuat thanh cong model tai: ../models/best_recipe_ranker.joblib")